# Building AI Agent

## Goal

This notebook is a hands-on journey to build an AI agent from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AI agent development.

---

## Version 9

In this version, we introduce multi-step execution.

The agent can now handle requests containing multiple sequential actions.

The Planner creates an ordered sequence of steps.

The Agent executes each step one at a time and can pass the result of one step to the next.

The LLM Router and LLM Parser remain unchanged.

The Tool Registry, Executor, and Memory Manager also remain unchanged.

The main goal of this version is to introduce sequential tool execution while preserving the existing architecture.

## 1. Imports

In [1]:
import re
import torch
import json

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
from huggingface_hub import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars

### Local Language Model

Version 9 uses the same small pretrained language model for routing and argument parsing.

The model runs locally inside the notebook.
It is downloaded from Hugging Face and does not require an external inference API.

In [2]:
# Local language model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()
disable_progress_bars()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

model.eval()

print(f"Model loaded on: {device}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded on: cpu


## 2. Tools

In [3]:
# Tools

def greeting(name):
    """Greet the given name."""
    return f"Hello {name.title()}, nice to meet you!"


def addition(a, b):
    """Add two numbers."""
    return a + b


def subtraction(a, b):
    """Subtract two numbers."""
    return a - b


def multiplication(a, b):
    """Multiply two numbers."""
    return a * b


def division(a, b):
    """Divide two numbers."""
    if b == 0:
        return "Error: division by zero!"
    return a / b


def power(a, b):
    """Raise a number to a power."""
    return a ** b

### Tool Registry

The Tool Registry provides a central place to store and access the tools available to the agent.

Instead of keeping tool metadata and synonyms in separate structures, each tool is registered together with its function, parameters, description, and synonyms.

The main goal is to centralize tool management while keeping the tools themselves unchanged.

In [4]:
# Tool registry

tool_registry = {}


def register_tool(name, function, parameters, description, synonyms):
    tool_registry[name] = {
        "function": function,
        "parameters": parameters,
        "description": description,
        "synonyms": synonyms
    }

In [5]:
# Register tools

register_tool(
    name="greeting",
    function=greeting,
    parameters=["name"],
    description="Greet the user by name.",
    synonyms=["hello", "hi", "hey", "good morning", "good afternoon", "good evening"]
)

register_tool(
    name="addition",
    function=addition,
    parameters=["a", "b"],
    description="Add two numbers.",
    synonyms=["add", "addition", "sum", "plus", "+"]
)

register_tool(
    name="subtraction",
    function=subtraction,
    parameters=["a", "b"],
    description="Subtract two numbers.",
    synonyms=["subtract", "subtraction", "minus", "take away", "-"]
)

register_tool(
    name="multiplication",
    function=multiplication,
    parameters=["a", "b"],
    description="Multiply two numbers.",
    synonyms=["multiply", "multiplication", "times", "*"]
)

register_tool(
    name="division",
    function=division,
    parameters=["a", "b"],
    description="Divide two numbers.",
    synonyms=["divide", "division", "divided by", "/"]
)

register_tool(
    name="power",
    function=power,
    parameters=["a", "b"],
    description="Raise a number to a power.",
    synonyms=["power", "raise", "raised to", "**"]
)

## 3. LLM Parser

In [6]:
# LLM Parser

def normalize_argument(value):
    if isinstance(value, str):
        if value.isdigit():
            return int(value)

        try:
            return float(value)
        except ValueError:
            return value

    return value

def build_parser_prompt(tool_name, request):
    tool_info = tool_registry.get(tool_name)

    parameters = ", ".join(tool_info["parameters"])

    return f"""
Extract the arguments required by the selected tool.

Selected tool:
{tool_name}

Tool description:
{tool_info["description"]}

Required parameters:
{parameters}

User request:
{request}

Extract argument values only from the user request.
Do not use the tool name, tool description, or parameter names as argument values.

Return only a valid JSON object.

Use exactly the required parameter names.
Infer the corresponding values from the user request.
Use numbers as numbers and text as strings.
If a required value cannot be determined from the user request, use null.

Do not explain your answer.
"""


def parser(tool_name, request):
    if tool_name is None:
        return {}

    tool_info = tool_registry.get(tool_name)

    if not tool_info:
        return {}

    prompt = build_parser_prompt(tool_name, request)

    messages = [
        {
            "role": "system",
            "content": "You are an argument parser. Extract the required tool arguments and return only JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    start = response.find("{")
    end = response.rfind("}") + 1

    if start == -1 or end == 0:
        return {}

    try:
        arguments = json.loads(response[start:end])
    except json.JSONDecodeError:
        return {}

    return {
        parameter: normalize_argument(arguments.get(parameter))
        for parameter in tool_info["parameters"]
    }

## 4. LLM Router

In [7]:
# LLM Router

def build_router_prompt(request):
    available_tools = []

    for tool_name, tool_info in tool_registry.items():
        available_tools.append(
            f"- {tool_name}: {tool_info['description']}"
        )

    tools_text = "\n".join(available_tools)
    allowed_answers = "\n".join(tool_registry.keys())

    return f"""
Select the tool that can actually handle the user request.

Available tools:
{tools_text}

User request:
{request}

Allowed answers:
{allowed_answers}
none

Select a tool only if it can directly perform the requested task.
If none of the available tools can handle the request, return none.
Do not select an unrelated tool just because it is the closest option.

Return only one allowed answer.
Do not explain your choice.
"""


def choose_tool(request):
    prompt = build_router_prompt(request)

    messages = [
        {
            "role": "system",
            "content": "You are a tool router. Select a tool only when it can handle the request. Otherwise output none."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

    for tool_name in tool_registry:
        if re.search(rf"\b{re.escape(tool_name)}\b", response):
            return tool_name

    return None

## 5. Planner

In [8]:
# Planner

def split_request(request):
    return [
        step.strip(" ,.")
        for step in request.split(" then ")
        if step.strip()
    ]


def planner(request):
    requests = split_request(request)

    plan = []

    for step_request in requests:
        tool_name = choose_tool(step_request)

        if tool_name is None:
            plan.append({
                "request": step_request,
                "tool": None,
                "reason": "No suitable tool was found for the request.",
                "needs_arguments": []
            })
            continue

        tool_info = tool_registry.get(tool_name)

        plan.append({
            "request": step_request,
            "tool": tool_name,
            "reason": tool_info["description"],
            "needs_arguments": tool_info["parameters"]
        })

    return plan

In [9]:
def resolve_step_request(request, previous_result):
    if previous_result is None:
        return request

    return request.replace("the result", str(previous_result))

## 6. Executor

In [10]:
# Validate arguments

def arguments_are_valid(arguments):
    if not arguments:
        return False

    for value in arguments.values():
        if value is None:
            return False

    return True

In [11]:
# Executor

def executor(tool_name, arguments):
    tool_info = tool_registry.get(tool_name)

    if not tool_info:
        return "Tool not found!"

    if not arguments_are_valid(arguments):
        return "Invalid arguments"

    function = tool_info["function"]

    return function(**arguments)

## 7. Memory Manager

In [12]:
# Memory manager

memory = []


def save_to_memory(state):
    memory.append(state)


def get_memory():
    return memory


def get_last_interaction():
    if not memory:
        return None

    return memory[-1]


def clear_memory():
    memory.clear()

## 8. Agent

In [13]:
# Agent

def agent(request):
    state = {
        "request": request,
        "plan": [],
        "steps": [],
        "result": None
    }

    plan = planner(request)
    state["plan"] = plan

    previous_result = None

    for step in plan:
        action = step["tool"]

        step_state = {
            "request": step["request"],
            "action": action,
            "arguments": {},
            "result": None
        }

        if action is None:
            step_state["result"] = "I cannot handle this request yet."
            state["steps"].append(step_state)
            state["result"] = step_state["result"]
            break

        resolved_request = resolve_step_request(
            step["request"],
            previous_result
        )

        arguments = parser(action, resolved_request)
        step_state["arguments"] = arguments

        result = executor(action, arguments)
        step_state["result"] = result

        state["steps"].append(step_state)

        previous_result = result
        state["result"] = result

    save_to_memory(state)

    return state

## 9. Tests

In [14]:
clear_memory()

In [15]:
tool_registry

{'greeting': {'function': <function __main__.greeting(name)>,
  'parameters': ['name'],
  'description': 'Greet the user by name.',
  'synonyms': ['hello',
   'hi',
   'hey',
   'good morning',
   'good afternoon',
   'good evening']},
 'addition': {'function': <function __main__.addition(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Add two numbers.',
  'synonyms': ['add', 'addition', 'sum', 'plus', '+']},
 'subtraction': {'function': <function __main__.subtraction(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Subtract two numbers.',
  'synonyms': ['subtract', 'subtraction', 'minus', 'take away', '-']},
 'multiplication': {'function': <function __main__.multiplication(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Multiply two numbers.',
  'synonyms': ['multiply', 'multiplication', 'times', '*']},
 'division': {'function': <function __main__.division(a, b)>,
  'parameters': ['a', 'b'],
  'description': 'Divide two numbers.',
  'synonyms': ['divide', 'division', 

In [16]:
agent("What is 2 + 3?")

{'request': 'What is 2 + 3?',
 'plan': [{'request': 'What is 2 + 3?',
   'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']}],
 'steps': [{'request': 'What is 2 + 3?',
   'action': 'addition',
   'arguments': {'a': 2, 'b': 3},
   'result': 5}],
 'result': 5}

In [17]:
agent("Hello, my name is luca.")

{'request': 'Hello, my name is luca.',
 'plan': [{'request': 'Hello, my name is luca',
   'tool': 'greeting',
   'reason': 'Greet the user by name.',
   'needs_arguments': ['name']}],
 'steps': [{'request': 'Hello, my name is luca',
   'action': 'greeting',
   'arguments': {'name': 'luca'},
   'result': 'Hello Luca, nice to meet you!'}],
 'result': 'Hello Luca, nice to meet you!'}

In [18]:
agent("What is the weather in Rome?")

{'request': 'What is the weather in Rome?',
 'plan': [{'request': 'What is the weather in Rome?',
   'tool': None,
   'reason': 'No suitable tool was found for the request.',
   'needs_arguments': []}],
 'steps': [{'request': 'What is the weather in Rome?',
   'action': None,
   'arguments': {},
   'result': 'I cannot handle this request yet.'}],
 'result': 'I cannot handle this request yet.'}

In [19]:
agent("Could you calculate the product of 2 and 3?")

{'request': 'Could you calculate the product of 2 and 3?',
 'plan': [{'request': 'Could you calculate the product of 2 and 3?',
   'tool': 'multiplication',
   'reason': 'Multiply two numbers.',
   'needs_arguments': ['a', 'b']}],
 'steps': [{'request': 'Could you calculate the product of 2 and 3?',
   'action': 'multiplication',
   'arguments': {'a': 2, 'b': 3},
   'result': 6}],
 'result': 6}

In [20]:
agent("Multiply three by four.")

{'request': 'Multiply three by four.',
 'plan': [{'request': 'Multiply three by four',
   'tool': 'multiplication',
   'reason': 'Multiply two numbers.',
   'needs_arguments': ['a', 'b']}],
 'steps': [{'request': 'Multiply three by four',
   'action': 'multiplication',
   'arguments': {'a': 3, 'b': 4},
   'result': 12}],
 'result': 12}

In [21]:
agent("Add 2 and 3, then multiply the result by 4.")

{'request': 'Add 2 and 3, then multiply the result by 4.',
 'plan': [{'request': 'Add 2 and 3',
   'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']},
  {'request': 'multiply the result by 4',
   'tool': 'multiplication',
   'reason': 'Multiply two numbers.',
   'needs_arguments': ['a', 'b']}],
 'steps': [{'request': 'Add 2 and 3',
   'action': 'addition',
   'arguments': {'a': 2, 'b': 3},
   'result': 5},
  {'request': 'multiply the result by 4',
   'action': 'multiplication',
   'arguments': {'a': 5, 'b': 4},
   'result': 20}],
 'result': 20}

In [22]:
get_memory()

[{'request': 'What is 2 + 3?',
  'plan': [{'request': 'What is 2 + 3?',
    'tool': 'addition',
    'reason': 'Add two numbers.',
    'needs_arguments': ['a', 'b']}],
  'steps': [{'request': 'What is 2 + 3?',
    'action': 'addition',
    'arguments': {'a': 2, 'b': 3},
    'result': 5}],
  'result': 5},
 {'request': 'Hello, my name is luca.',
  'plan': [{'request': 'Hello, my name is luca',
    'tool': 'greeting',
    'reason': 'Greet the user by name.',
    'needs_arguments': ['name']}],
  'steps': [{'request': 'Hello, my name is luca',
    'action': 'greeting',
    'arguments': {'name': 'luca'},
    'result': 'Hello Luca, nice to meet you!'}],
  'result': 'Hello Luca, nice to meet you!'},
 {'request': 'What is the weather in Rome?',
  'plan': [{'request': 'What is the weather in Rome?',
    'tool': None,
    'reason': 'No suitable tool was found for the request.',
    'needs_arguments': []}],
  'steps': [{'request': 'What is the weather in Rome?',
    'action': None,
    'arguments':

In [23]:
get_last_interaction()

{'request': 'Add 2 and 3, then multiply the result by 4.',
 'plan': [{'request': 'Add 2 and 3',
   'tool': 'addition',
   'reason': 'Add two numbers.',
   'needs_arguments': ['a', 'b']},
  {'request': 'multiply the result by 4',
   'tool': 'multiplication',
   'reason': 'Multiply two numbers.',
   'needs_arguments': ['a', 'b']}],
 'steps': [{'request': 'Add 2 and 3',
   'action': 'addition',
   'arguments': {'a': 2, 'b': 3},
   'result': 5},
  {'request': 'multiply the result by 4',
   'action': 'multiplication',
   'arguments': {'a': 5, 'b': 4},
   'result': 20}],
 'result': 20}

## Notes

Some test cases were intentionally designed to verify specific parts of the agent.

- `"Hello, my name is luca."` checks that the LLM Router selects the greeting tool and the LLM Parser extracts the name.
- `"Could you calculate the product of 2 and 3?"` checks that the LLM Router understands the user's intent without depending on manually defined routing synonyms.
- `"Multiply three by four."` checks that the LLM Parser can extract numeric arguments expressed as words instead of digits.
- `"What is the weather in Rome?"` checks that the LLM Router returns no tool when none of the registered tools can handle the request.
- `"Add 2 and 3, then multiply the result by 4."` checks that the Planner creates multiple steps and that the Agent passes the result of one step to the next.
- The Tool Registry provides tool names and descriptions to the LLM Router.
- The Tool Registry provides the selected tool description and required parameters to the LLM Parser.
- The same local language model is used for routing and argument parsing.
- The Planner remains rule-based.
- The Executor and Memory Manager remain unchanged.
- The agent can now perform multiple actions sequentially, one step at a time.
- The language model runs locally inside the notebook and does not use an external inference API.

Future versions will introduce new components and gradually evolve the architecture.